<a href="https://colab.research.google.com/github/apolusouza/robo-de-cortes/blob/main/sistema_corte_automatico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install moviepy

In [2]:
from moviepy.editor import VideoFileClip, AudioClip, CompositeVideoClip, ImageClip, TextClip

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



In [3]:
!apt update
!apt install imagemagick -y

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,765 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,228 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:13 http://security.ubuntu.com/ubuntu jam

In [4]:
from moviepy.video.VideoClip import VideoClip
from os import write
import textwrap

In [14]:
import moviepy.config as cfg

cfg.change_settings({
    "IMAGEMAGICK_BINARY": "/usr/bin/convert"
})

In [15]:
WIDTH = 1080
HEIGHT = 900

video = VideoFileClip("video.mp4")
tempo = video.duration
print(tempo)
print(f'Altura:{video.h}')
print(f'Largura:{video.w}')

if video.h > HEIGHT:
  print(f'Ele é maior que {HEIGHT}')

78.9
Altura:1920
Largura:1080
Ele é maior que 900


In [12]:
!which convert

/usr/bin/convert


In [16]:
texto = "Python é uma linguagem de programação de alto nível, amplamente utilizada para desenvolvimento web, análise de dados e inteligência artificial."

# Quebra automaticamente em linhas de no máximo 40 caracteres
texto_quebrado = textwrap.fill(texto, width=40)
print(texto_quebrado)

Python é uma linguagem de programação de
alto nível, amplamente utilizada para
desenvolvimento web, análise de dados e
inteligência artificial.


In [17]:
if len(texto)>=100 and len(texto)<=150:
     texto_quebrado = textwrap.fill(texto,30)
else:
  print('e-mail enviado')

In [19]:
!sed -i 's/<policy domain="path" rights="none" pattern="@\*"/<!-- disabled -->/g' /etc/ImageMagick-6/policy.xml

In [20]:
text_clip = TextClip(
                     txt=texto_quebrado,
                     fontsize=50,
                     color='white',
                     font='Arial'

).set_duration(tempo)

In [21]:
def calculate_xy_ct(element):
    x_center = element.w/2
    y_center = element.h/2
    xy_center = [x_center,y_center]
    return xy_center

def cropped_video(element, xy_center, key_side, measure):
    # Para usar x_center/y_center, o MoviePy precisa que width E height sejam definidos.
    # Começamos com os valores atuais do elemento e sobrescrevemos o que queremos mudar.
    kwargs = {
        'x_center': xy_center[0],
        'y_center': xy_center[1],
        'width': element.w,
        'height': element.h
    }
    kwargs[key_side] = measure
    return element.crop(**kwargs)

#Largura maior que 1080 ou menor
if video.w < WIDTH or video.w > WIDTH:
  width_norm = video.resize(width=WIDTH)
  if width_norm.h > HEIGHT:
    xy_center = calculate_xy_ct(width_norm)
    video = cropped_video(width_norm, xy_center, 'height', HEIGHT)
elif video.h < HEIGHT: #Altura menor que 400
  height_norm = video.resize(height=HEIGHT)
  if height_norm.w > WIDTH:
    xy_center = calculate_xy_ct(height_norm)
    video = cropped_video(height_norm,  xy_center, 'width', WIDTH)
elif video.h > HEIGHT: # Altura maior que 400
  print(f'Ele É maior que {HEIGHT}')
  xy_center = calculate_xy_ct(video)
  print(xy_center[0])
  video = cropped_video(video, xy_center, 'height', HEIGHT)

print(f'Altura: {video.h}, Largura:{video.w}')

Ele É maior que 900
540.0
Altura: 900, Largura:1080


In [22]:
imagem = ImageClip("imagem.jpg").set_duration(tempo)

In [23]:
position_text = text_clip.set_position((0,344))
espace_li = text_clip.size[1] +394
print(espace_li)

779


In [24]:
li_video = video.set_position((0,espace_li))
print(video.h)
print(video.w)

900
1080


In [25]:
juntando = CompositeVideoClip([imagem, position_text,li_video])
print(juntando.pos)
juntando.write_videofile("novo_video.mp4")

<function VideoClip.__init__.<locals>.<lambda> at 0x7af68c53e980>
Moviepy - Building video novo_video.mp4.
MoviePy - Writing audio in novo_videoTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video novo_video.mp4



t: 100%|█████████▉| 2365/2367 [04:57<00:00, 10.23it/s, now=None]WARNING:py.warnings:/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:123: UserWarning: Warning: in file video.mp4, 6220800 bytes wanted but 0 bytes read,at frame 2365/2368, at time 78.83/78.90 sec. Using the last valid frame instead.
  warnings.warn("Warning: in file %s, "%(self.filename)+

t: 100%|██████████| 2367/2367 [04:57<00:00,  9.92it/s, now=None]WARNING:py.warnings:/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:123: UserWarning: Warning: in file video.mp4, 6220800 bytes wanted but 0 bytes read,at frame 2366/2368, at time 78.87/78.90 sec. Using the last valid frame instead.
  warnings.warn("Warning: in file %s, "%(self.filename)+



Moviepy - Done !
Moviepy - video ready novo_video.mp4
